In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time
import os

In [2]:
path="/mnt/c/Users/isabe/OneDrive/Desktop/Lavoro/Tuscia/chiara/peaks_background/bufale_isa/"
os.chdir(path)

In [3]:
file_path_robot = './dati_bufale/dati_grezzi_agosto_luglio/Dataset_CH4/Bufale_Rinna/Luglio/Dati_robot/Mungiture_1-31_luglio.csv'
file_path_bufale = './dati_bufale/dati_grezzi_agosto_luglio/Dataset_CH4/Bufale_Rinna/Luglio/protocollo_1/ch4_moologger_detrended_linear.csv'

In [ ]:
#test dati luglio agosto (impossibile senza dati robot)

In [ ]:
#gestire dati agosto, formato diverso

In [4]:
#file_path_bufale = "./dati_bufale/dati_grezzi_agosto_luglio/Dataset_CH4/Bufale_Rinna/Luglio/Dati_moologger/luglio_detrended.csv"

df_sniffer = pd.read_csv(file_path_bufale, usecols=['timestamp', 'CH4', 'CH4-mgm3', 'CO2', 'CO2-mgm3'])
df_sniffer.head()

,timestamp,CH4,CH4-mgm3,CO2,CO2-mgm3
0,07/01/2024 09:51:00.024698,56.772196,128.452,645.1,1161.186
1,07/01/2024 09:51:01.026654,46.172326,121.498,627.0,1128.606
2,07/01/2024 09:51:02.029196,56.772457,128.452,602.3,1084.145
3,07/01/2024 09:51:03.032440,49.772587,123.860,605.6,1090.085
4,07/01/2024 09:51:04.035013,52.772718,125.828,611.2,1100.166


In [5]:
df_sniffer['timestamp'] = pd.to_datetime(df_sniffer['timestamp'])
df_sniffer['date'] = [d.date() for d in df_sniffer['timestamp']]
df_sniffer['time'] = [d.time() for d in df_sniffer['timestamp']]

In [23]:
df_sniffer.tail()

,timestamp,CH4,CH4-mgm3,CO2,CO2-mgm3,date,time
2212279,2024-08-30 08:20:42.685681,-67.571255,59.896,969.1,1741.869,2024-08-30,08:20:42.685681
2212280,2024-08-30 08:20:43.688536,-75.971283,54.385,929.1,1672.388,2024-08-30,08:20:43.688536
2212281,2024-08-30 08:20:44.691011,-83.771312,49.268,873.3,1571.948,2024-08-30,08:20:44.691011
2212282,2024-08-30 08:20:45.693816,-87.871341,46.579,836.3,1505.348,2024-08-30,08:20:45.693816
2212283,2024-08-30 08:20:46.695530,-94.471370,42.249,819.5,1475.107,2024-08-30,08:20:46.695530


In [7]:
df_sniffer.shape

(2212284, 7)

In [ ]:
type(df_sniffer['date'][1])

In [6]:
date_str = '2024-07-01'
start_date = datetime.strptime(date_str, '%Y-%m-%d').date()
type(start_date)
df_sniffer = df_sniffer[df_sniffer['date'] >= start_date]
#df2 = df[df['Date'].dt.strftime('%Y-%m-%d') == "2021-10-08"]


In [ ]:
df_sniffer.tail()

In [7]:
# Caricamento dei dati dal file CSV del robot di mungitura
df_robot = pd.read_csv(file_path_robot, header=1, sep=";", usecols=["Ora inizio", "Numero animale",
                                "Azione", "Durata (mm:ss)",
                                "Produzione (kg)", "Numero mungitura"])
df_robot['ID_animal'] = df_robot['Numero animale']
df_robot = df_robot.drop(columns = ['Numero animale'])

In [8]:
df_robot.tail()

,Ora inizio,Azione,Durata (mm:ss),Produzione (kg),Numero mungitura,ID_animal
5965,31/07/2024 22:40,Mungitura,13:49,"0,03",3.0,245581
5966,31/07/2024 22:54,Rifiutata,00:00,NaN,NaN,9076
5967,31/07/2024 23:12,Mungitura,06:53,"2,11",3.0,234554
5968,31/07/2024 23:25,Rifiutata,00:00,NaN,NaN,5968
5969,31/07/2024 23:54,Rifiutata,00:00,NaN,NaN,7854


In [11]:
df_robot.shape

(5378, 6)

In [9]:
# Filtraggio dei record con "Status" uguale a "Avvenuto con successo"
df_robot = df_robot[df_robot['Azione'] == 'Mungitura']

# Conversione e arrotondamento della colonna 'timestamp' al secondo più vicino nel DataFrame del Mologger
df_sniffer['timestamp'] = pd.to_datetime(df_sniffer['timestamp'], errors='coerce')

In [ ]:
type(df_robot['Durata (mm:ss)'][1])

In [ ]:
'''
# Convert 'Durata (mm:ss)' to string in 'mm:ss' format

df_robot['Durata (mm:ss)'] = df_robot['Durata (mm:ss)'].apply(lambda x: x.strftime('%M:%S')
df_robot['Ora inizio'] = df_robot['Ora inizio'].apply(lambda x: x.strftime('%Y-%m-%d %H:%M'))

#print(df_robot.dtypes)
'''

In [10]:
df_robot['Milking_duration'] = df_robot['Durata (mm:ss)']
df_robot['Duration (hh:mm:ss)'] = '00:' + df_robot['Milking_duration']
df_robot['Duration (hh:mm:ss)'] = pd.to_timedelta(df_robot['Duration (hh:mm:ss)'])

In [11]:
df_robot['Milking_duration'] = df_robot['Milking_duration'].apply(lambda x: str(x))

In [14]:
df_robot.head()

,Ora inizio,Azione,Durata (mm:ss),Produzione (kg),Numero mungitura,ID_animal,Milking_duration,Duration (hh:mm:ss)
0,01/08/2024 00:06,Mungitura,10:36,"5,16",1.0,245584,10:36,0 days 00:10:36
1,01/08/2024 00:18,Mungitura,06:27,"3,86",1.0,245575,06:27,0 days 00:06:27
5,01/08/2024 00:49,Mungitura,08:51,"5,13",1.0,7854,08:51,0 days 00:08:51
6,01/08/2024 00:59,Mungitura,09:11,"2,59",1.0,245578,09:11,0 days 00:09:11
7,01/08/2024 01:17,Mungitura,08:07,"3,70",1.0,245577,08:07,0 days 00:08:07


In [15]:
df_robot.shape

(2204, 8)

In [12]:
#df_robot['Ora inizio'] = df_robot['Ora inizio'].apply(lambda x: x.strptime('%Y/%m/%d %H:%M:%S'))

df_robot['Ora inizio'] = pd.to_datetime(df_robot['Ora inizio'], format='%d/%m/%Y %H:%M')

In [13]:
df_robot_sorted = df_robot.sort_values('Ora inizio')

#df_robot_sorted=df_robot
df_robot_sorted['Ora inizio'] = pd.to_datetime(df_robot_sorted['Ora inizio'])

#associazione della durata di mungitura e ingresso bufala a misurazione sniffer con taglio del dato all'uscita dell'animale
df_robot_sorted['Milking_end'] = None
df_robot_sorted['Milking_end'] = df_robot_sorted['Ora inizio'] + df_robot_sorted['Duration (hh:mm:ss)']

In [14]:
#####
             
# Unione dei DataFrame df_robot e df_mologger basata sulla data con tolleranza di 1 minuto
# pulisci il timestamp per valori nan o assenti
df_sniffer.dropna(subset=['timestamp'], inplace=True)
df_sniffer.dropna(subset=['CH4'], inplace=True)

merged_df = pd.merge_asof(df_sniffer, df_robot_sorted, left_on='timestamp', right_on='Ora inizio')

merged_df=merged_df[merged_df['timestamp'] < merged_df['Milking_end']]

#merged_df.drop(columns=['Gruppo',	'MS/MP', 'CC(*1000 cellule/ml)',	'Kick-off',	'Incompleto',	
#                        'Capezzoli non trovati',	'Mammella',	'Capezzolo',	'AS.1',	'AD.1',	
#                        'PS.1',	'PD.1',	'AS.2',	'AD.2',	'PS.2',	'PD.2',	'AS.3',	'AD.3',	'PS.3',	
#                        'PD.3',	'Utente',	'Destinazione latte',	'Motivo separazione',	'AS.4',	
#                        'AD.4',	'PS.4',	'PD.4',	'AS',	'AD',	'PS',	'PD'], inplace=True)

na_count = merged_df['CH4'].value_counts().get('N.A.', 0)
print("Count of 'N.A.' in the column CH4:", na_count)


Count of 'N.A.' in the column CH4: 0


In [15]:

#print("Data types of all columns:")
#print(merged_df.dtypes)

merged_df['CH4'] = pd.to_numeric(merged_df['CH4'], errors='coerce')

merged_df.replace(',', '.', regex=True, inplace=True)
merged_df.replace(':', '.', regex=True, inplace=True)
merged_df['Milking_duration'] = pd.to_numeric(merged_df['Milking_duration'], errors='coerce')
merged_df['Produzione (kg)'] = pd.to_numeric(merged_df['Produzione (kg)'], errors='coerce')

#print("Data types of all columns:")
#print(merged_df.dtypes)

#DO NOT perform if data are DETRENDED
#merged_df=merged_df[merged_df['CH4'] > 0]

#merged_df=merged_df[merged_df['CH4'] < 5000]

#output_file = './dati_bufale/merged_buffalo_sniffer_AMS.csv'
#merged_df.to_csv(output_file, index=False)

In [21]:
print(df_sniffer.shape)
print(merged_df.head())
print(merged_df.shape)

(2212284, 7)
                   timestamp         CH4  CH4-mgm3    CO2  CO2-mgm3  \
0 2024-08-01 09:25:28.260847 -218.968072   -81.283  600.5  1080.905   
1 2024-08-01 09:25:29.263544 -221.268101   -82.792  607.6  1093.685   
2 2024-08-01 09:25:30.265996 -223.568129   -84.301  604.4  1087.925   
3 2024-08-01 09:25:31.268802 -221.068158   -82.661  605.3  1089.725   
4 2024-08-01 09:25:32.271264 -220.068187   -82.005  607.0  1092.605   

         date             time          Ora inizio     Azione Durata (mm:ss)  \
0  2024-08-01  09:25:28.260847 2024-08-01 09:20:00  Mungitura          06.09   
1  2024-08-01  09:25:29.263544 2024-08-01 09:20:00  Mungitura          06.09   
2  2024-08-01  09:25:30.265996 2024-08-01 09:20:00  Mungitura          06.09   
3  2024-08-01  09:25:31.268802 2024-08-01 09:20:00  Mungitura          06.09   
4  2024-08-01  09:25:32.271264 2024-08-01 09:20:00  Mungitura          06.09   

   Produzione (kg)  Numero mungitura  ID_animal  Milking_duration  \
0         

In [16]:
###CONTA MUNGITURE PER PERIODO
# Initialize 'Milking_number' column
merged_df['Milking_number'] = 0

# Initialize variables
milking_count = 0
previous_id = None

# Iterate through the DataFrame
for index, row in merged_df.iterrows():
    current_id = row['ID_animal']
   
    if current_id != previous_id:
        milking_count += 1
   
    # Update the 'Milking_number' column
    merged_df.at[index, 'Milking_number'] = milking_count
   
    # Update previous_id for the next iteration
    previous_id = current_id
    

output_file = './dati_bufale/dati_grezzi_agosto_luglio/Dataset_CH4/Bufale_Rinna/Luglio/protocollo_1/ch4_merged_sniffer_AMS_detrended.csv'
merged_df.to_csv(output_file, index=False)

In [ ]:
merged_df[1860:1870]

In [17]:
merged_df.tail(15)

,timestamp,CH4,CH4-mgm3,CO2,CO2-mgm3,date,time,Ora inizio,Azione,Durata (mm:ss),Produzione (kg),Numero mungitura,ID_animal,Milking_duration,Duration (hh:mm:ss),Milking_end,Milking_number
1886660,2024-07-23 12:42:21.845882,15.719713,-59.830,848.0,1526.408,2024-07-23,12:42:21.845882,2024-07-23 12:38:00,Mungitura,04.33,3.07,2.0,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
1886661,2024-07-23 12:42:22.848522,19.919844,-57.075,849.7,1529.468,2024-07-23,12:42:22.848522,2024-07-23 12:38:00,Mungitura,04.33,3.07,2.0,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
1886662,2024-07-23 12:42:23.851711,33.919974,-47.169,966.9,1740.429,2024-07-23,12:42:23.851711,2024-07-23 12:38:00,Mungitura,04.33,3.07,2.0,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
1886663,2024-07-23 12:42:24.854658,100.320104,-4.330,1485.4,2673.733,2024-07-23,12:42:24.854658,2024-07-23 12:38:00,Mungitura,04.33,3.07,2.0,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
1886664,2024-07-23 12:42:25.857463,151.420235,29.194,1931.6,3476.897,2024-07-23,12:42:25.857463,2024-07-23 12:38:00,Mungitura,04.33,3.07,2.0,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
1886665,2024-07-23 12:42:26.860076,8.420365,-64.620,1981.5,3566.718,2024-07-23,12:42:26.860076,2024-07-23 12:38:00,Mungitura,04.33,3.07,2.0,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
1886666,2024-07-23 12:42:27.861799,97.320495,-6.298,1830.2,3285.376,2024-07-23,12:42:27.861799,2024-07-23 12:38:00,Mungitura,04.33,3.07,2.0,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
1886667,2024-07-23 12:42:28.864713,101.420626,-3.608,1681.1,3025.995,2024-07-23,12:42:28.864713,2024-07-23 12:38:00,Mungitura,04.33,3.07,2.0,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
1886668,2024-07-23 12:42:29.867926,82.320756,-16.138,1518.4,2733.134,2024-07-23,12:42:29.867926,2024-07-23 12:38:00,Mungitura,04.33,3.07,2.0,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
1886669,2024-07-23 12:42:30.871007,57.320887,-32.539,1322.2,2368.992,2024-07-23,12:42:30.871007,2024-07-23 12:38:00,Mungitura,04.33,3.07,2.0,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
